# Reviewer 3 — Comment 7
## Protein-side FG20 redundancy / discriminability audit

This notebook executes the 20 historical SMARTS patterns on the 20 standard amino-acid structures and measures zero columns, distinct vectors, entropy, and collision groups. The cleanup rules mirror the historical metadata cleanup used in the audit.

In [1]:
from collections import defaultdict, Counter
from math import log2
import numpy as np
from rdkit import Chem, rdBase

FG=(('CarboxylicAcid','[CX3](=O)[OX2H1]'),('Ester','[CX3](=O)[OX2][#6]'),('Amide','[NX3][CX3](=O)[#6]'),('Anhydride','[CX3](=O)O[CX3](=O)'),('AcylHalide','[CX3](=O)[Cl,Br,I,F]'),('Aldehyde','[CX3H1](=O)[#6]'),('Ketone','[#6][CX3](=O)[#6]'),('Alcohol','[#6;!a][OX2H]'),('Phenol','c[OX2H]'),('Ether','[OX2]([#6])[#6]'),('Nitrile','[CX2]#N'),('Nitro','[$([NX3](=O)=O),$([NX3+](=O)[O-])]'),('Amine_Primary','[NX3;H2][#6]'),('Amine_Secondary','[NX3;H1]([#6])[#6]'),('Amine_Tertiary','[NX3]([#6])([#6])[#6]'),('Thiol','[#16X2H]'),('Thioether','[#16X2]([#6])[#6]'),('Sulfoxide','[#16X3](=O)([#6])[#6]'),('Sulfone','[#16X4](=O)(=O)([#6])[#6]'),('Aryl','c1ccccc1'))
N=tuple(x[0] for x in FG)
P=[(n,Chem.MolFromSmarts(s)) for n,s in FG]
AA={'A':'NCC(C)C(=O)O','R':'NCC(CCCNC(N)=N)C(=O)O','N':'NCC(C(=O)N)C(=O)O','D':'NCC(C(=O)O)C(=O)O','C':'NCC(S)C(=O)O','E':'NCC(CCC(=O)O)C(=O)O','Q':'NCC(CCC(=O)N)C(=O)O','G':'NCC(=O)O','H':'NCC(Cc1c[nH]cn1)C(=O)O','I':'NCC(C(C)CC)C(=O)O','L':'NCC(CC(C)C)C(=O)O','K':'NCC(CCCCN)C(=O)O','M':'NCC(CCSC)C(=O)O','F':'NCC(Cc1ccccc1)C(=O)O','P':'N1CCC(C(=O)O)C1','S':'NCC(CO)C(=O)O','T':'NCC(C(O)C)C(=O)O','W':'NCC(Cc1c2ccccc2[nH]c1)C(=O)O','Y':'NCC(Cc1ccc(O)cc1)C(=O)O','V':'NCC(C(C)C)C(=O)O'}

def groups(m):
    g={n for n,p in P if m.HasSubstructMatch(p)}
    if 'Ester' in g:g.discard('Ether')
    if 'CarboxylicAcid' in g:g.discard('Alcohol')
    if 'Phenol' in g:g.discard('Alcohol')
    return g

rows=[]
for aa,smi in AA.items():
    g=groups(Chem.MolFromSmiles(smi)); v=np.array([int(n in g) for n in N]); rows.append((aa,smi,v))
mat=np.stack([v for _,_,v in rows])
zero=[N[i] for i in range(20) if mat[:,i].sum()==0]
clusters=defaultdict(list)
for aa,_,v in rows: clusters[tuple(v.tolist())].append(aa)
counts=Counter(tuple(v.tolist()) for _,_,v in rows)
ent=-sum((c/20)*log2(c/20) for c in counts.values())
summary={'rdkit_version':rdBase.rdkitVersion,'zero_columns_count':len(zero),'zero_columns':zero,'distinct_vectors':len(clusters),'max_categorical_capacity_bits_log2_distinct':log2(len(clusters)),'empirical_shannon_entropy_uniform_residues_bits':ent,'collision_groups':[x for x in clusters.values() if len(x)>1]}
summary

{'rdkit_version': '2025.09.4',
 'zero_columns_count': 12,
 'zero_columns': ['Ester',
  'Anhydride',
  'AcylHalide',
  'Aldehyde',
  'Ketone',
  'Alcohol',
  'Ether',
  'Nitrile',
  'Nitro',
  'Amine_Tertiary',
  'Sulfoxide',
  'Sulfone'],
 'distinct_vectors': 8,
 'max_categorical_capacity_bits_log2_distinct': 3.0,
 'empirical_shannon_entropy_uniform_residues_bits': 2.219240704636849,
 'collision_groups': [['A', 'D', 'E', 'G', 'H', 'I', 'L', 'K', 'S', 'T', 'V'],
  ['N', 'Q'],
  ['F', 'W']]}

In [2]:
import json
print(json.dumps(summary, indent=2))

{
  "rdkit_version": "2025.09.4",
  "zero_columns_count": 12,
  "zero_columns": [
    "Ester",
    "Anhydride",
    "AcylHalide",
    "Aldehyde",
    "Ketone",
    "Alcohol",
    "Ether",
    "Nitrile",
    "Nitro",
    "Amine_Tertiary",
    "Sulfoxide",
    "Sulfone"
  ],
  "distinct_vectors": 8,
  "max_categorical_capacity_bits_log2_distinct": 3.0,
  "empirical_shannon_entropy_uniform_residues_bits": 2.219240704636849,
  "collision_groups": [
    [
      "A",
      "D",
      "E",
      "G",
      "H",
      "I",
      "L",
      "K",
      "S",
      "T",
      "V"
    ],
    [
      "N",
      "Q"
    ],
    [
      "F",
      "W"
    ]
  ]
}


In [3]:
for sig, aas in sorted(clusters.items(), key=lambda x:(-len(x[1]), x[1])):
    active=[N[i] for i,x in enumerate(sig) if x]
    print(''.join(aas), '->', active)

ADEGHILKSTV -> ['CarboxylicAcid', 'Amine_Primary']
FW -> ['CarboxylicAcid', 'Amine_Primary', 'Aryl']
NQ -> ['CarboxylicAcid', 'Amide', 'Amine_Primary']
C -> ['CarboxylicAcid', 'Amine_Primary', 'Thiol']
M -> ['CarboxylicAcid', 'Amine_Primary', 'Thioether']
P -> ['CarboxylicAcid', 'Amine_Secondary']
R -> ['CarboxylicAcid', 'Amine_Primary', 'Amine_Secondary']
Y -> ['CarboxylicAcid', 'Phenol', 'Amine_Primary', 'Aryl']
